In [0]:
%fs ls "abfss://unity-catalog-storage@dbstorageqh3nvwxzzfwa6.dfs.core.windows.net/7405605315193329/mnt/DE-Associate-Book/datasets/school/"

In [0]:
%sql
USE SCHEMA de_associate_school;
DESCRIBE EXTENDED students_mv

# Capturing Data Changes
Change data capture is a process that identifies and captures changes made to data in a data source and records those changes to target.
Captures three types of row-level changes:
- Insertions
- Updates
- Deletions


[SOURCE DB] -== CDC (inserts, updates, deletes) ====>  [TARGET DB]

## CDC Feed
Changes are logged at the source as events containing:
- Operation type
- Timestamp/version number

Originates from databases with built-in CDC and third-party software known as CDC agents
Delta Lake has built-in CDF.


## CDC Feed Delivery
- Data stream:
  - CDC events continuously streamed from the source to target (near real-time)
- JSON files:
  - CDC events captured and stored as JSON

## CDC in DLT
APPLY CHANGES INTO command allows one to apply changes from a CDC feed table into a DLT table, which serves as target table

```sql
  APPLY CHANGES INTO LIVE.target_table
  FROM STREAM(LIVE.cdc_feed_table)
  KEYS (key_field)
  APPLY AS DELETE WHEN operation_field = "DELETE"
  SEQUENCE BY sequence_field
  COLUMNS *
```

Advantages
- Reduced code complexity
- Automatic ordering of late-arriving records
- Performing upsert operations
- Flexible delete handling
- Multiple key fields
- Excluding columns
- Built-in support for slowly changing dimensions

Disadvantages
- Breaking append-only req for streaming sources

## Processing Change Data Capture
We will modify our school DLT pipeline to add CDC processing capabilities

In [0]:
%run ./School-Setup

In [0]:
load_new_json_data()

In [0]:
%sql
SELECT * 
FROM json.`${dataset.school}/courses-cdc/02.json`

As noted above, INSERT AND UPDATE rows will have complete records, but DELETE records will have null

## Extending DLT Pipelines with New Notebooks
We are going to add a new notebook that will define additional DLT tables for processing the courses data.